# ⚙️ Step 4: Model Training, Evaluation, and World Cup Simulation
---
This notebook serves as our unified machine learning pipeline. Instead of running a background script, we execute each stage of the data lifecycle here:
1. **Data Ingestion**: Loading and cleaning historical international match data.
2. **Feature Engineering**: Chronological computation of team form, goal differences, and rolling metrics.
3. **Model Selection**: Benchmarking Logistic Regression, Random Forests, and XGBoost using Stratified 5-Fold Cross-Validation.
4. **Simulation**: Using our finalized model to forecast the entire opening round of the FIFA World Cup 2026.

In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# --- THE FIX FOR GOOGLE COLAB WIDGETS ---
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    print("✅ Colab Widget Manager Enabled!")
except ImportError:
    pass # Running locally or on Binder
# ---------------------------------------

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
DATA_URL = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"

def load_and_prepare_data() -> pd.DataFrame:
    """Loads historical data from local data folder or downloads it if missing."""
    csv_path = DATA_DIR / "international_results.csv"
    
    if csv_path.exists():
        print("📂 Dataset detected in local cache. Loading...")
        df = pd.read_csv(csv_path)
    else:
        print("📡 Local file missing. Fetching live dataset from remote repository...")
        df = pd.read_csv(DATA_URL)
        df.to_csv(csv_path, index=False)
        print("💾 Download complete. Cached locally.")
            
    # Clean, parse, and filter for modern era matches
    df['date'] = pd.to_datetime(df['date'])
    df = df[df['date'] >= '2000-01-01'].reset_index(drop=True)
    df = df.dropna(subset=['home_score', 'away_score'])
    
    # Ensure types are correct, specifically turning boolean 'neutral' into 0 or 1
    df['home_score'] = df['home_score'].astype(int)
    df['away_score'] = df['away_score'].astype(int)
    df['neutral'] = df['neutral'].astype(int) 
    
    print(f"⚽ Successfully prepared {len(df):,} international matches (2000–Present).")
    return df

raw_data = load_and_prepare_data()

📂 Dataset detected in local cache. Loading...
⚽ Successfully prepared 25,316 international matches (2000–Present).


In [4]:
class FeatureEngineer:
    """Computes running statistics chronologically to eliminate data leakage."""
    def __init__(self, window: int = 5):
        self.window = window
        self.team_history = {}

    def _update_history(self, team: str, goals_for: int, goals_against: int, pts: int):
        if team not in self.team_history:
            self.team_history[team] = {'gf': [], 'ga': [], 'pts': [], 'wins': []}
        self.team_history[team]['gf'].append(goals_for)
        self.team_history[team]['ga'].append(goals_against)
        self.team_history[team]['pts'].append(pts)
        self.team_history[team]['wins'].append(1 if pts == 3 else 0)

    def _get_form_features(self, team: str, prefix: str) -> dict:
        hist = self.team_history.get(team, {'gf': [], 'ga': [], 'pts': [], 'wins': []})
        recent_gf = hist['gf'][-self.window:]
        recent_ga = hist['ga'][-self.window:]
        recent_pts = hist['pts'][-self.window:]
        recent_wins = hist['wins'][-self.window:]
        
        n = len(recent_gf) if len(recent_gf) > 0 else 1
        return {
            f'{prefix}_avg_goals_last{self.window}': sum(recent_gf) / n,
            f'{prefix}_avg_conceded_last{self.window}': sum(recent_ga) / n,
            f'{prefix}_points_last{self.window}': sum(recent_pts),
            f'{prefix}_win_rate_last{self.window}': sum(recent_wins) / n,
            f'{prefix}_goal_diff_last{self.window}': sum(recent_gf) - sum(recent_ga)
        }

    def build_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.sort_values('date').reset_index(drop=True)
        features_list = []

        for idx, row in df.iterrows():
            home, away = row['home_team'], row['away_team']
            gh, ga = row['home_score'], row['away_score']
            
            # Extract features BEFORE updating state with today's match results
            match_feats = {**self._get_form_features(home, 'home'), **self._get_form_features(away, 'away')}
            features_list.append(match_feats)
            
            h_pts, a_pts = (3, 0) if gh > ga else ((1, 1) if gh == ga else (0, 3))
            self._update_history(home, gh, ga, h_pts)
            self._update_history(away, ga, gh, a_pts)

        return pd.concat([df, pd.DataFrame(features_list)], axis=1)

print("⚙️ Engineering feature transformers...")
engineer = FeatureEngineer(window=5)
engineered_df = engineer.build_features(raw_data)
print("✅ Feature space built successfully.")

⚙️ Engineering feature transformers...
✅ Feature space built successfully.


In [5]:
# Create targets (0: Away Win, 1: Draw, 2: Home Win)
conditions = [
    (engineered_df['home_score'] < engineered_df['away_score']),
    (engineered_df['home_score'] == engineered_df['away_score']),
    (engineered_df['home_score'] > engineered_df['away_score'])
]
choices = [0, 1, 2]  
engineered_df['outcome'] = np.select(conditions, choices, default=1)

# Isolate feature set AND include the neutral flag
feature_cols = [col for col in engineered_df.columns if 'last5' in col]
feature_cols.append('neutral') # Model learns Home Advantage vs Neutral ground

X = engineered_df[feature_cols]
y = engineered_df['outcome']

print(f"📊 Dataset shapes: X={X.shape}, y={y.shape}")

# Finalize and serialize the best-performing model (XGBoost)
print("\n💾 Training final XGBoost engine on complete historical dataset...")
final_model = xgb.XGBClassifier(eval_metric='mlogloss', random_state=42)
final_model.fit(X, y)

# Save the model
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)
model_output_path = MODEL_DIR / "fifa_xgboost.joblib"
joblib.dump(final_model, model_output_path)
print(f"✅ Production model serialized and saved to: {model_output_path}")

📊 Dataset shapes: X=(25316, 11), y=(25316,)

💾 Training final XGBoost engine on complete historical dataset...
✅ Production model serialized and saved to: models\fifa_xgboost.joblib


In [6]:
from IPython.display import display

# Official 2026 Group Stage Matrix
WC2026_MATCHES = [
    # Group A
    ("Mexico", "South Africa", "2026-06-11"), ("South Korea", "Czechia", "2026-06-11"),
    ("Mexico", "Czechia", "2026-06-17"), ("South Korea", "South Africa", "2026-06-17"),
    ("Mexico", "South Korea", "2026-06-22"), ("Czechia", "South Africa", "2026-06-22"),
    # Group B
    ("Canada", "Bosnia and Herzegovina", "2026-06-12"), ("Qatar", "Switzerland", "2026-06-13"),
    ("Canada", "Qatar", "2026-06-18"), ("Bosnia and Herzegovina", "Switzerland", "2026-06-18"),
    ("Canada", "Switzerland", "2026-06-23"), ("Bosnia and Herzegovina", "Qatar", "2026-06-23"),
    # Group C
    ("Brazil", "Morocco", "2026-06-13"), ("Haiti", "Scotland", "2026-06-13"),
    ("Brazil", "Scotland", "2026-06-18"), ("Morocco", "Haiti", "2026-06-19"),
    ("Brazil", "Haiti", "2026-06-23"), ("Scotland", "Morocco", "2026-06-24"),
    # Group D
    ("USA", "Paraguay", "2026-06-12"), ("Australia", "Turkey", "2026-06-13"),
    ("USA", "Australia", "2026-06-19"), ("Paraguay", "Turkey", "2026-06-19"),
    ("USA", "Turkey", "2026-06-25"), ("Paraguay", "Australia", "2026-06-25")
]

# Fetch the absolute latest historical metrics for each team
latest_state = engineered_df.sort_values('date').groupby('home_team').last().reset_index()

simulation_rows = []

for home, away, date in WC2026_MATCHES:
    if home in latest_state['home_team'].values and away in latest_state['home_team'].values:
        
        # Build vector out of latest forms
        h_feats = {f'home_{k.split("home_")[1]}': v for k, v in latest_state[latest_state['home_team'] == home].iloc[0].to_dict().items() if 'home_avg' in k or 'home_point' in k or 'home_win' in k or 'home_goal' in k}
        a_feats = {f'away_{k.split("home_")[1]}': v for k, v in latest_state[latest_state['home_team'] == away].iloc[0].to_dict().items() if 'home_avg' in k or 'home_point' in k or 'home_win' in k or 'home_goal' in k}
        
        # --- HOST COUNTRY LOGIC ---
        # 0 (False) if it's a host country playing at home, 1 (True) for all other neutral matches
        is_neutral = 0 if home in ['USA', 'Mexico', 'Canada'] else 1
        
        combined_feats = {**h_feats, **a_feats}
        combined_feats['neutral'] = is_neutral
        
        match_vector = pd.DataFrame([combined_feats])[feature_cols]
        probs = final_model.predict_proba(match_vector)[0]
        
        # Extract individual probabilities
        team2_win_prob = probs[0]
        draw_prob = probs[1]
        team1_win_prob = probs[2]
        
        # Dynamically assign the winner's actual name
        if np.argmax(probs) == 2:
            predicted_winner = home
        elif np.argmax(probs) == 0:
            predicted_winner = away
        else:
            predicted_winner = "Draw"
        
        simulation_rows.append({
            "Date": date,
            "Team 1": home,
            "Team 2": away,
            f"{home} Win Prob": f"{team1_win_prob*100:.1f}%",
            "Draw Prob": f"{draw_prob*100:.1f}%",
            f"{away} Win Prob": f"{team2_win_prob*100:.1f}%",
            "Predicted Outcome": predicted_winner
        })

sim_df = pd.DataFrame(simulation_rows)
display(sim_df)

,Date,Team 1,Team 2,Mexico Win Prob,Draw Prob,South Africa Win Prob,Predicted Outcome,South Korea Win Prob,Canada Win Prob,Bosnia and Herzegovina Win Prob,Qatar Win Prob,Switzerland Win Prob,Brazil Win Prob,Morocco Win Prob,Haiti Win Prob,Scotland Win Prob,Australia Win Prob,Turkey Win Prob,Paraguay Win Prob
0,2026-06-11,Mexico,South Africa,71.4%,15.9%,12.6%,Mexico,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-06-17,South Korea,South Africa,NaN,28.6%,26.4%,South Korea,45.1%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-06-22,Mexico,South Korea,51.3%,32.0%,NaN,Mexico,16.7%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-06-12,Canada,Bosnia and Herzegovina,NaN,25.8%,NaN,Canada,NaN,50.4%,23.8%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-06-13,Qatar,Switzerland,NaN,25.8%,NaN,Switzerland,NaN,NaN,NaN,21.7%,52.6%,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2026-06-18,Canada,Qatar,NaN,14.1%,NaN,Canada,NaN,80.7%,NaN,5.2%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2026-06-18,Bosnia and Herzegovina,Switzerland,NaN,22.5%,NaN,Bosnia and Herzegovina,NaN,NaN,41.1%,NaN,36.5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2026-06-23,Canada,Switzerland,NaN,20.2%,NaN,Canada,NaN,60.6%,NaN,NaN,19.2%,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2026-06-23,Bosnia and Herzegovina,Qatar,NaN,24.6%,NaN,Bosnia and Herzegovina,NaN,NaN,63.1%,12.3%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2026-06-13,Brazil,Morocco,NaN,33.9%,NaN,Morocco,NaN,NaN,NaN,NaN,NaN,27.0%,39.2%,NaN,NaN,NaN,NaN,NaN


In [5]:
# Benchmark Models using 5-Fold Stratified Cross-Validation
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    'XGBoost': xgb.XGBClassifier(eval_metric='mlogloss', random_state=42)
}

print("🏋️ Evaluating candidate models...")
for name, model in models.items():
    cv_results = cross_validate(model, X, y, cv=StratifiedKFold(5), scoring=['accuracy', 'f1_macro'])
    print(f" ↳ {name:20} -> Mean Accuracy: {cv_results['test_accuracy'].mean():.4f} | F1-Score: {cv_results['test_f1_macro'].mean():.4f}")

# Finalize and serialize the best-performing model (XGBoost)
print("\n💾 Training final XGBoost engine on complete historical dataset...")
final_model = xgb.XGBClassifier(eval_metric='mlogloss', random_state=42)
final_model.fit(X, y)

model_output_path = MODEL_DIR / "fifa_xgboost.joblib"
joblib.dump(final_model, model_output_path)
print(f"✅ Production model serialized and saved to: {model_output_path}")

🏋️ Evaluating candidate models...
 ↳ Logistic Regression  -> Mean Accuracy: 0.5275 | F1-Score: 0.3660
 ↳ Random Forest        -> Mean Accuracy: 0.5249 | F1-Score: 0.3563
 ↳ XGBoost              -> Mean Accuracy: 0.5125 | F1-Score: 0.3774

💾 Training final XGBoost engine on complete historical dataset...
✅ Production model serialized and saved to: models\fifa_xgboost.joblib


## 📅 World Cup 2026 Simulation
We extract the final operational state of each team at the end of our dataset timeline and use our serialized model to predict the upcoming tournament group match schedule.

In [11]:
from IPython.display import display

# Official 2026 Group Stage Matrix
# Official 2026 Group Stage Matchday 1 (All 48 Teams)
WC2026_MATCHES = [
    # Group A
    ("Mexico", "South Africa", "2026-06-11"), 
    ("South Korea", "Czech Republic", "2026-06-11"), # Note: Kaggle usually uses 'Czech Republic' instead of 'Czechia'
    # Group B
    ("Canada", "Bosnia and Herzegovina", "2026-06-12"), 
    ("Qatar", "Switzerland", "2026-06-13"),
    # Group C
    ("Brazil", "Morocco", "2026-06-13"), 
    ("Haiti", "Scotland", "2026-06-13"),
    # Group D
    ("USA", "Paraguay", "2026-06-12"), 
    ("Australia", "Turkey", "2026-06-13"), # Note: Kaggle usually uses 'Turkey' instead of 'Türkiye'
    # Group E
    ("Germany", "Curaçao", "2026-06-14"), 
    ("Ivory Coast", "Ecuador", "2026-06-14"),
    # Group F
    ("Netherlands", "Japan", "2026-06-14"), 
    ("Sweden", "Tunisia", "2026-06-14"),
    # Group G
    ("Belgium", "Egypt", "2026-06-15"), 
    ("Iran", "New Zealand", "2026-06-15"),
    # Group H
    ("Spain", "Cabo Verde", "2026-06-15"), 
    ("Saudi Arabia", "Uruguay", "2026-06-15"),
    # Group I
    ("France", "Senegal", "2026-06-16"), 
    ("Iraq", "Norway", "2026-06-16"),
    # Group J
    ("Argentina", "Algeria", "2026-06-16"), 
    ("Austria", "Jordan", "2026-06-16"),
    # Group K
    ("Portugal", "DR Congo", "2026-06-17"), 
    ("Uzbekistan", "Colombia", "2026-06-17"),
    # Group L
    ("England", "Croatia", "2026-06-17"), 
    ("Ghana", "Panama", "2026-06-17")

]

# Fetch the absolute latest historical metrics for each team
latest_state = engineered_df.sort_values('date').groupby('home_team').last().reset_index()

simulation_rows = []

for home, away, date in WC2026_MATCHES:
    if home in latest_state['home_team'].values and away in latest_state['home_team'].values:
        
        # Build vector out of latest forms
        h_feats = {f'home_{k.split("home_")[1]}': v for k, v in latest_state[latest_state['home_team'] == home].iloc[0].to_dict().items() if 'home_avg' in k or 'home_point' in k or 'home_win' in k or 'home_goal' in k}
        a_feats = {f'away_{k.split("home_")[1]}': v for k, v in latest_state[latest_state['home_team'] == away].iloc[0].to_dict().items() if 'home_avg' in k or 'home_point' in k or 'home_win' in k or 'home_goal' in k}
        
        # --- HOST COUNTRY LOGIC ---
        # 0 (False) if it's a host country playing at home, 1 (True) for all other neutral matches
        is_neutral = 0 if home in ['USA', 'Mexico', 'Canada'] else 1
        
        combined_feats = {**h_feats, **a_feats}
        combined_feats['neutral'] = is_neutral
        
        match_vector = pd.DataFrame([combined_feats])[feature_cols]
        probs = final_model.predict_proba(match_vector)[0]
        
        # Extract individual probabilities
        team2_win_prob = probs[0]
        draw_prob = probs[1]
        team1_win_prob = probs[2]
        
        # Dynamically assign the winner's actual name
        if np.argmax(probs) == 2:
            predicted_winner = home
        elif np.argmax(probs) == 0:
            predicted_winner = away
        else:
            predicted_winner = "Draw"
        
        simulation_rows.append({
            "Date": date,
            "Team 1": home,
            "Team 2": away,
            f"{home} Win Prob": f"{team1_win_prob*100:.1f}%",
            "Draw Prob": f"{draw_prob*100:.1f}%",
            f"{away} Win Prob": f"{team2_win_prob*100:.1f}%",
            "Predicted Outcome": predicted_winner
        })

sim_df = pd.DataFrame(simulation_rows)
display(sim_df)
          
      

,Date,Team 1,Team 2,Mexico Win Prob,Draw Prob,South Africa Win Prob,Predicted Outcome,South Korea Win Prob,Czech Republic Win Prob,Canada Win Prob,...,Austria Win Prob,Jordan Win Prob,Portugal Win Prob,DR Congo Win Prob,Uzbekistan Win Prob,Colombia Win Prob,England Win Prob,Croatia Win Prob,Ghana Win Prob,Panama Win Prob
0,2026-06-11,Mexico,South Africa,71.4%,15.9%,12.6%,Mexico,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-06-11,South Korea,Czech Republic,NaN,21.8%,NaN,South Korea,39.1%,39.1%,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-06-12,Canada,Bosnia and Herzegovina,NaN,25.8%,NaN,Canada,NaN,NaN,50.4%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-06-13,Qatar,Switzerland,NaN,25.8%,NaN,Switzerland,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-06-13,Brazil,Morocco,NaN,33.9%,NaN,Morocco,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2026-06-13,Haiti,Scotland,NaN,23.9%,NaN,Haiti,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2026-06-13,Australia,Turkey,NaN,23.7%,NaN,Turkey,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2026-06-14,Germany,Curaçao,NaN,4.3%,NaN,Germany,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2026-06-14,Ivory Coast,Ecuador,NaN,18.1%,NaN,Ivory Coast,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2026-06-14,Netherlands,Japan,NaN,30.3%,NaN,Japan,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
display(sim_df.head(100))

,Date,Team 1,Team 2,Mexico Win Prob,Draw Prob,South Africa Win Prob,Predicted Outcome,South Korea Win Prob,Canada Win Prob,Bosnia and Herzegovina Win Prob,Qatar Win Prob,Switzerland Win Prob,Brazil Win Prob,Morocco Win Prob,Haiti Win Prob,Scotland Win Prob,Australia Win Prob,Turkey Win Prob,Paraguay Win Prob
0,2026-06-11,Mexico,South Africa,71.4%,15.9%,12.6%,Mexico,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-06-17,South Korea,South Africa,NaN,28.6%,26.4%,South Korea,45.1%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-06-22,Mexico,South Korea,51.3%,32.0%,NaN,Mexico,16.7%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-06-12,Canada,Bosnia and Herzegovina,NaN,25.8%,NaN,Canada,NaN,50.4%,23.8%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-06-13,Qatar,Switzerland,NaN,25.8%,NaN,Switzerland,NaN,NaN,NaN,21.7%,52.6%,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2026-06-18,Canada,Qatar,NaN,14.1%,NaN,Canada,NaN,80.7%,NaN,5.2%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2026-06-18,Bosnia and Herzegovina,Switzerland,NaN,22.5%,NaN,Bosnia and Herzegovina,NaN,NaN,41.1%,NaN,36.5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2026-06-23,Canada,Switzerland,NaN,20.2%,NaN,Canada,NaN,60.6%,NaN,NaN,19.2%,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2026-06-23,Bosnia and Herzegovina,Qatar,NaN,24.6%,NaN,Bosnia and Herzegovina,NaN,NaN,63.1%,12.3%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2026-06-13,Brazil,Morocco,NaN,33.9%,NaN,Morocco,NaN,NaN,NaN,NaN,NaN,27.0%,39.2%,NaN,NaN,NaN,NaN,NaN
